In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
RAW_DIR = "../../data/raw"
OUT_DIR = "../../data/python_master"

In [ ]:
with pd.ExcelFile(f"{RAW_DIR}/OBR/efo-march-2026-detailed-forecast-tables-economy.xlsx") as xls:
    hs_raw = pd.read_excel(xls, sheet_name='1.16', skiprows=2, index_col=1,
                       nrows=93)
    rate_raw = pd.read_excel(xls, sheet_name='1.9', skiprows=2, index_col=1,
                         nrows = 93)
    cpi_raw = pd.read_excel(xls, sheet_name='1.7', skiprows=3, index_col=1,
                        nrows=93)



for df in [hs_raw, rate_raw, cpi_raw]:
    df.drop(columns="Unnamed: 0", inplace=True)

hs = hs_raw.rename(columns={
    'House price index \n(Jan 2023 = 100)': "hprice",
    'Residential property transactions \n(000s, seasonally adjusted)': "vol",
})[["hprice", "vol"]]
rate = rate_raw.rename(columns={"Bank Rate": "r3"})["r3"]
cpi  = cpi_raw["CPI.1"].rename("CPI")

proc = pd.concat([hs, rate, cpi], axis=1).reset_index().rename(columns={"index": "period"})
proc['period'] = pd.PeriodIndex(proc['period'], freq='Q')

# Filtering to observation period end - CHANGE THIS WHEN NEW DATA RELEASED
proc = proc[proc["period"] >= "2025Q4"].reset_index(drop=True)

# Parsing construction costs
url = "https://costmodelling.com/construction-indices"
cc_html = BeautifulSoup(requests.get(url).text, "html.parser")
rows = cc_html.find("table", class_="indices").find_all("tr")[1:]
cc = pd.DataFrame([[c.text.strip() for c in r.find_all('td')[:3]] for r in rows],
                  columns=["date", "tpi", "bci"])
cc['date'] = cc['date'].replace('', pd.NA).ffill()
cc['quarter'] = cc.groupby('date').cumcount() + 1
cc['period'] = pd.PeriodIndex(cc['date'] + 'Q' + cc['quarter'].astype(str), freq='Q')
cc['bci'] = pd.to_numeric(cc['bci'], errors='coerce')
proc = proc.merge(cc[["period", "bci"]], on='period', how='left')

# Beyond last real bci hold index flat
last_real = proc['bci'].last_valid_index()
for i in range(last_real + 1, len(proc)):
    proc.loc[i, 'bci'] = proc.loc[i-1, 'bci'] * (proc.loc[i, 'CPI'] / proc.loc[i-1, 'CPI'])


# Taking growth paths
proc["dlrprc"] = np.log(proc["hprice"]).diff() - np.log(proc["CPI"]).diff()
proc["dlvol"]  = np.log(proc["vol"]).diff()
proc["dlrcc"]  = np.log(proc["bci"]).diff()    - np.log(proc["CPI"]).diff()

# Taking last observations 2025Q4 in england data
eng = pd.read_csv("../../data/python_master/england_master.csv")
last = eng.iloc[-2] 
lrprc_anchor = np.log(last['hprice'] / last['p_def'])
lvol_anchor  = np.log(last['vol'])
lrcc_anchor  = np.log(last['cc']    / last['gdp_def'])

# Building England series
proc["lrprc"] = lrprc_anchor + proc["dlrprc"].fillna(0).cumsum()
proc["lvol"]  = lvol_anchor  + proc["dlvol"].fillna(0).cumsum()
proc["lrcc"]  = lrcc_anchor  + proc["dlrcc"].fillna(0).cumsum()

out = proc[["period", "lrprc", "lvol", "r3", "lrcc"]]
out.to_csv(f"{OUT_DIR}/OBR/obr_scenario.csv", index=False)

In [ ]:
out